In [1]:
import pandas as pd
import numpy as np
from db_queries import get_covariate_estimates, get_location_metadata

In [ ]:
data_path = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_10_07.01/data.parquet"
no_cutoff_path = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_09_23.01/data_no_10_yr_cutoff.parquet"

In [3]:
df = pd.read_parquet(data_path)
# df.drop(columns=["index"], inplace=True)
df.head()

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_3_m,age_6_m,age_12_m,age_24_m,age_36_m,age_48_m,age_60_m,age_group,lower_bound,upper_bound
0,MDG,338,338,23,2,35,-16.418806,49.823825,75955,493562_338_35_2_1_2005_3_2021_5_194_1_31_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
1,TJK,307,307,7,3,35,38.467862,68.826338,87095,341838_307_35_3_1_2009_4_2017_9_101_1_NA_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
2,BEN,539,539,17,6,8,6.754141,1.807671,82042,79839_539_8_6_1_1989_9_2011_1_268_1_NA_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
3,ARM,267,267,17,2,1,39.549752,45.964419,90891,218563_267_1_2_1_1993_9_2015_1_268_1_NA_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
4,NPL,343,343,13,1,42,28.654069,81.076586,93790,286782_343_42_1_2_2015_9_2016_8_11_1_19_NA,...,0,0,0,0,0,0,0,age_1_m,0,1


In [6]:
[c for c in df.columns]

['ihme_loc_id',
 'geospatial_id',
 'psu',
 'strata',
 'line_id',
 'hh_id',
 'lat',
 'long',
 'lbd_admin2_id',
 'indv_id',
 'year_start',
 'year_end',
 'nid',
 'int_year',
 'int_month',
 'sex_id',
 'age_month',
 'pweight',
 'birth_year',
 'birth_month',
 'int_birth_year_diff_months',
 'age_month_original',
 'int_year_original',
 'int_month_original',
 'any_days_over_30C',
 'child_alive',
 'child_mortality',
 'mean_temperature_cumul',
 'days_over_30C_cumul',
 'total_precipitation_cumul',
 'elevation_cumul',
 'consumption_cumul',
 'consumption_pd_cumul',
 'days_over_24C_monthly_cumul',
 'days_over_25C_monthly_cumul',
 'days_over_26C_monthly_cumul',
 'days_over_27C_monthly_cumul',
 'days_over_28C_monthly_cumul',
 'days_over_29C_monthly_cumul',
 'days_over_30C_monthly_cumul',
 'days_over_31C_monthly_cumul',
 'days_over_32C_monthly_cumul',
 'mean_temperature_monthly_cumul',
 'total_precipitation_monthly_cumul',
 'q75_monthly_cumul',
 'q80_monthly_cumul',
 'q85_monthly_cumul',
 'q90_monthly_c

In [21]:
REQUIRED_COLS = [
    "child_mortality",
    "ihme_loc_id",
    "sex_id",
    "consumption_median_RT",
    "consumption_pd_cumul",
    "days_over_30C_monthly_cumul",
    "total_precipitation_monthly_cumul",
    "age_1_m",
    "age_3_m",
    "age_6_m",
    "age_12_m",
    "age_24_m",
    "age_36_m",
    "age_48_m",
    "age_60_m",
]
print(len(REQUIRED_COLS))

15


In [9]:
df["age_group"].unique()

array(['age_1_m', 'age_3_m', 'age_6_m', 'age_12_m', 'age_24_m',
       'age_36_m', 'age_48_m', 'age_60_m'], dtype=object)

In [ ]:
# Previously data contained extra rows following childrens' deaths
df[
    df["indv_id"]
    == df[(df["age_group"] == "age_12_m") & (df["child_mortality"] == 1)][
        "indv_id"
    ].iloc[0]
]

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_3_m,age_6_m,age_12_m,age_24_m,age_36_m,age_48_m,age_60_m,age_group,lower_bound,upper_bound
5598483,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,0,0,0,0,0,0,0,age_1_m,0,1
10277610,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,1,0,0,0,0,0,0,age_3_m,1,3
20386639,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,0,1,0,0,0,0,0,age_6_m,3,6
23372903,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,0,0,1,0,0,0,0,age_12_m,6,12


In [14]:
cols_to_check = [
    "child_mortality",
    "consumption_pd_cumul",
    "int_birth_year_diff_months",
    "birth_year",
    "days_over_30C_monthly_cumul",
    "total_precipitation_cumul",
    "lat",
    "long",
]

for c in cols_to_check:
    print(f"{c}: {df[c].isna().sum()} missing values")

child_mortality: 0 missing values
consumption_pd_cumul: 0 missing values
int_birth_year_diff_months: 0 missing values
birth_year: 0 missing values
days_over_30C_monthly_cumul: 0 missing values
total_precipitation_cumul: 0 missing values
lat: 0 missing values
long: 0 missing values


Add median consumption_pd for each country-birth_year

In [ ]:
rt_consumption_df = pd.read_parquet(
    "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/ldi/v8/national_estimates.parquet"
)
rt_consumption_df = rt_consumption_df[
    (rt_consumption_df["scenario"] == "reference")
    & (rt_consumption_df["population_percentile"] == 0.5)
]
rt_consumption_df = rt_consumption_df[
    ["ihme_loc_id", "year_id", "location_id", "ldipc"]
]
rt_consumption_df.rename(
    columns={"year_id": "birth_year", "ldipc": "consumption_median_RT"},
    inplace=True,
)

In [17]:
df = df.merge(rt_consumption_df, on=["ihme_loc_id", "birth_year"], how="left")
assert len(df[df["consumption_median_RT"].isna()]) == 0
df["consumption_median_RT"] = df["consumption_median_RT"] / 365

Cut off int_birth_year_diff_months to max 10

In [18]:
df.to_parquet(no_cutoff_path, index=False)

In [ ]:
f"{len(df)-24400000}"

'1572326'

In [ ]:
len(df[df["int_birth_year_diff_months"] >= 108])

2682693

In [22]:
df["int_birth_year_diff_months"].value_counts()

 73    278630
 74    272902
 75    271226
 76    268405
 77    262284
        ...  
 0      20144
-9      15977
-10      7411
-11      4209
-12       380
Name: int_birth_year_diff_months, Length: 132, dtype: int64

In [29]:
df = df[df["int_birth_year_diff_months"] <= 108]
print(f"Number of rows after filtering: {len(df):,}")
df.to_parquet(data_path, index=False)

Number of rows after filtering: 23,515,189
